# H3 Turbo Performance Benchmarks
This notebook compares the performance of `h3-turbo` (GPU) against `h3-py` + `numpy` (CPU).

In [ ]:
# No wheel provided, assuming h3_turbo is installed.
# !pip install h3_turbo

In [ ]:
# Kernel/Environment Fix for RunPod/Docker
import sys
import os
import glob

print(f'Python Executable: {sys.executable}')

try:
    import h3_turbo
    import h3_turbo._h3_turbo
    print('h3_turbo is already importable and working.')
except ImportError:
    print('h3_turbo not importable, adding development and venv paths...')
    # 1. Try to find the venv site-packages dynamically
    venv_patterns = ['/opt/venv/lib/python*/site-packages', '/app/.venv*/lib/python*/site-packages', '/workspace/.venv*/lib/python*/site-packages']
    found_paths = []
    for pattern in venv_patterns:
        found_paths.extend(glob.glob(pattern))

    for p in found_paths:
        if p not in sys.path:
            sys.path.append(p)
            print(f'Added venv path: {p}')

    # 2. Add local build/src directories
    local_build = os.path.join(os.getcwd(), 'build')
    if os.path.exists(local_build) and local_build not in sys.path:
        sys.path.append(local_build)
        print(f'Added local build path: {local_build}')
    local_src = os.path.join(os.getcwd(), 'src')
    if os.path.exists(local_src) and local_src not in sys.path:
        sys.path.append(local_src)
        print(f'Added local src path: {local_src}')


In [ ]:
# Environment Setup for AdaptiveCpp
import os

# Configure environment to help AdaptiveCpp find its backends and reduce noise
acpp_paths = ['/usr/local/lib/hipSYCL', '/app/AdaptiveCpp/lib', '/usr/local/lib']
current_ld = os.environ.get('LD_LIBRARY_PATH', '')
for p in acpp_paths:
    if os.path.exists(p) and p not in current_ld:
        current_ld = p + ':' + current_ld
os.environ['LD_LIBRARY_PATH'] = current_ld
# Set visibility mask to avoid trying missing backends
import platform
if platform.system() != 'Darwin':
    os.environ['ACPP_VISIBILITY_MASK'] = os.environ.get('ACPP_VISIBILITY_MASK', 'cuda;omp;hip')
os.environ['ACPP_DEBUG_LEVEL'] = '1' # Only show errors (0=none, 1=error, 2=warn, 3=info)


In [ ]:
import h3_turbo
import numpy as np
import h3
import time
import os
try:
    import matplotlib.pyplot as plt
except ImportError:
    !pip install matplotlib
    import matplotlib.pyplot as plt

print(f'h3_turbo version: {getattr(h3_turbo, "__version__", "unknown")}')

In [ ]:
# Set License Key if present in environment
if 'H3_TURBO_LICENSE' in os.environ:
    h3_turbo.set_license_key(os.environ['H3_TURBO_LICENSE'].strip())
    print('License key set from environment.')

In [ ]:
# Check which device AdaptiveCpp is using
try:
    device = h3_turbo.device_name()
    print(f'Active Device: {device}')
    if any(vendor in device for vendor in ['NVIDIA', 'AMD', 'Intel', 'Tesla', 'GeForce', 'Radeon']):
        print('✅ GPU Backend Active')
    else:
        print('⚠️ Warning: Running on CPU/OpenMP Backend')
except AttributeError:
    print('h3_turbo.device_name() not available')

In [ ]:
def numpy_apply_weight(h3_array):
    """
    Vectorized implementation of the 50-loop scramble using NumPy.
    Matches the logic in RawPerformanceBenchmark.scala.
    """
    p = h3_array.astype(np.uint64)
    c1 = np.uint64(0xBF58476D1CE4E5B9)
    c2 = np.uint64(0x94D049BB133111EB)
    for _ in range(50):
        p ^= (p >> np.uint64(7))
        p *= c1
        p ^= (p >> np.uint64(13))
        p *= c2
        p ^= (p >> np.uint64(31))
    return p


In [ ]:
print('Warming up JIT...')
dummy_data = np.array([0x8928308280fffff], dtype=np.uint64)
h3_turbo.batch_transform(dummy_data, 5, scramble_iterations=50)
print('JIT Ready!')

## Benchmark 1: Raw Compute (Heatmap Logic)

In [ ]:
def run_raw_compute_benchmark(n=50000000):
    print(f'\n=== Raw Compute Benchmark (N={n:,}) ===')
    res_target = 9
    base_index = 0x8928308280fffff

    # Data Generation
    k = 100
    # Use h3_turbo for fast data generation
    root_cell = np.array([base_index], dtype=np.uint64)
    seeds_np = h3_turbo.grid_disk(root_cell, k)
    seeds_np = seeds_np[seeds_np != 0]
    repeats = (n // len(seeds_np)) + 1
    data_gpu = np.tile(seeds_np, repeats)[:n]
    np.random.shuffle(data_gpu)
    data_cpu = data_gpu.copy()

    # GPU Run
    print('Running GPU...')
    start_gpu = time.time()
    gpu_results = h3_turbo.batch_transform(data_gpu, res_target, scramble_iterations=50)
    gpu_duration = time.time() - start_gpu
    print(f'GPU Time: {gpu_duration:.4f} s | Throughput: {n/gpu_duration:,.0f} ops/s')

    # CPU Run
    import platform
    import os
    try:
        num_workers = len(os.sched_getaffinity(0))
    except AttributeError:
        num_workers = os.cpu_count() or 1
    print(f'Running CPU (Vectorized) on {platform.processor()} with {num_workers} vCPUs...')
    start_cpu = time.time()
    # Step 1: Cell to Parent (h3-py is scalar)
    parents_list = [h3.str_to_int(h3.cell_to_parent(h3.int_to_str(int(h)), res_target)) for h in data_cpu]
    parents_arr = np.array(parents_list, dtype=np.uint64)
    # Step 2: Scramble
    cpu_results = numpy_apply_weight(parents_arr)
    cpu_duration = time.time() - start_cpu
    print(f'CPU Time: {cpu_duration:.4f} s')

    print(f'Speedup: {cpu_duration / gpu_duration:.2f}x')
    if gpu_results is None:
        gpu_results = data_gpu
    is_equal = np.array_equal(gpu_results, cpu_results)
    if not is_equal:
        print('ERROR: Results do not match!')
        print(f'GPU Sample: {gpu_results[:5]}')
        print(f'CPU Sample: {cpu_results[:5]}')
        if np.array_equal(gpu_results, data_cpu) or np.all(gpu_results == 0):
            print('CRITICAL WARNING: GPU array is unmodified or all zeros! The kernel failed to launch, likely due to a GPU architecture mismatch.')
    assert is_equal, 'GPU and CPU results mismatch!'
    print('Verification Passed.')

run_raw_compute_benchmark()

## Benchmark 2: Spatial Join (Point-in-Polygon)

In [ ]:
import concurrent.futures
import multiprocessing

# Global for workers
_global_zone_set = None

def _init_worker(zone_set):
    global _global_zone_set
    _global_zone_set = zone_set

def _process_chunk_cpu(chunk, res_target):
    parents = [h3.str_to_int(h3.cell_to_parent(h3.int_to_str(int(p)), res_target)) for p in chunk]
    parents_np = np.array(parents, dtype=np.uint64)
    scrambled = numpy_apply_weight(parents_np)
    return np.array([1 if p in _global_zone_set else 0 for p in scrambled], dtype=np.uint8)

def run_spatial_join_benchmark(n_pings=50000000, n_zones=1000000):
    print(f'\n=== Spatial Join Benchmark (Pings={n_pings:,}, Zones={n_zones:,}) ===')
    res_target = 7
    base_index = 0x8928308280fffff

    # Data Generation
    # Use h3_turbo for fast data generation
    root_cell = np.array([base_index], dtype=np.uint64)
    pool_np = h3_turbo.grid_disk(root_cell, 200)
    pool_np = pool_np[pool_np != 0]
    zones = np.random.choice(pool_np, n_zones, replace=True)
    pings = pool_np[np.random.randint(0, len(pool_np), size=n_pings)]

    # GPU Run
    print('Running GPU...')
    start_gpu = time.time()
    gpu_results = h3_turbo.spatial_join(pings, zones, res_target, scramble_iterations=50)
    gpu_duration = time.time() - start_gpu
    print(f'GPU Time: {gpu_duration:.4f} s | Throughput: {n_pings/gpu_duration:,.0f} ops/s')

    # CPU Run
    try:
        num_workers = len(os.sched_getaffinity(0))
    except AttributeError:
        num_workers = os.cpu_count() or 1
    import platform
    print(f'Running CPU (Multi-process on {num_workers} vCPUs, {platform.processor()})...')
    # Pre-calculate zone set
    zone_parents = numpy_apply_weight(np.array([h3.str_to_int(h3.cell_to_parent(h3.int_to_str(int(z)), res_target)) for z in zones], dtype=np.uint64))
    zone_set = set(zone_parents)

    start_cpu = time.time()
    chunk_size = 1_000_000
    chunks = [pings[i:i + chunk_size] for i in range(0, len(pings), chunk_size)]
    with concurrent.futures.ProcessPoolExecutor(max_workers=num_workers, initializer=_init_worker, initargs=(zone_set,)) as executor:
        results = list(executor.map(_process_chunk_cpu, chunks, [res_target]*len(chunks)))
    cpu_results = np.concatenate(results)
    cpu_duration = time.time() - start_cpu
    print(f'CPU Time: {cpu_duration:.4f} s')

    print(f'Speedup: {cpu_duration / gpu_duration:.2f}x')
    is_equal = np.array_equal(gpu_results, cpu_results)
    if not is_equal:
        print('ERROR: Results do not match!')
        print(f'GPU Sample: {gpu_results[:5]}')
        print(f'CPU Sample: {cpu_results[:5]}')
    assert is_equal, 'GPU and CPU results mismatch!'
    print('Verification Passed.')

run_spatial_join_benchmark()

## Benchmark 3: Persistent Joiner vs spatial_join (Cache Reuse)

In [ ]:
def run_persistent_joiner_benchmark(n_pings=10000000, n_zones=100000):
    print(f'\n=== PersistentJoiner vs spatial_join Cache Reuse Benchmark ===')
    print(f'Total Pings: {n_pings:,} | Zones: {n_zones:,}')
    n_batches = 10
    batch_size = n_pings // n_batches
    res_target = 7
    base_index = 0x8928308280fffff

    # Data Generation
    root_cell = np.array([base_index], dtype=np.uint64)
    pool_np = h3_turbo.grid_disk(root_cell, 200)
    pool_np = pool_np[pool_np != 0]
    zones = np.random.choice(pool_np, n_zones, replace=True)
    pings = pool_np[np.random.randint(0, len(pool_np), size=n_pings)]

    # Divide pings into batches
    batches = [pings[i:i + batch_size] for i in range(0, n_pings, batch_size)]

    # 1. Baseline: spatial_join (0% cache hit / rebuild every time)
    print('Running spatial_join (0% cache hit)...')
    start = time.time()
    for batch in batches:
        _ = h3_turbo.spatial_join(batch, zones, res_target, scramble_iterations=50)
    time_spatial_join = time.time() - start
    print(f'spatial_join (0% hit rate): {time_spatial_join:.4f} s | Throughput: {n_pings/time_spatial_join:,.0f} pings/s')

    # 2. PersistentJoiner with 100% cache hit rate (build once, reuse always)
    print('Running PersistentJoiner (100% cache hit)...')
    start = time.time()
    joiner = h3_turbo.PersistentJoiner(zones, res_target, scramble_iterations=50)
    for batch in batches:
        results = np.zeros(len(batch), dtype=np.uint8)
        joiner.join(batch, results)
    time_100_hit = time.time() - start
    print(f'PersistentJoiner (100% hit rate): {time_100_hit:.4f} s | Throughput: {n_pings/time_100_hit:,.0f} pings/s')

    # 3. PersistentJoiner with 50% cache hit rate (rebuild every second batch)
    print('Running PersistentJoiner (50% cache hit)...')
    start = time.time()
    joiner = None
    for idx, batch in enumerate(batches):
        if idx % 2 == 0 or joiner is None:
            joiner = h3_turbo.PersistentJoiner(zones, res_target, scramble_iterations=50)
        results = np.zeros(len(batch), dtype=np.uint8)
        joiner.join(batch, results)
    time_50_hit = time.time() - start
    print(f'PersistentJoiner (50% hit rate): {time_50_hit:.4f} s | Throughput: {n_pings/time_50_hit:,.0f} pings/s')

    # Plotting
    scenarios = ['spatial_join\n(0% cache hit)', 'PersistentJoiner\n(50% cache hit)', 'PersistentJoiner\n(100% cache hit)']
    times = [time_spatial_join, time_50_hit, time_100_hit]
    throughputs = [n_pings / t for t in times]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#ff4444', '#ffaa00', '#00cc66']
    
    ax1.bar(scenarios, times, color=colors, edgecolor='black', alpha=0.8)
    ax1.set_ylabel('Execution Time (seconds)', fontsize=12)
    ax1.set_title('Execution Time Comparison (Lower is Better)', fontsize=13)
    for i, v in enumerate(times):
        ax1.text(i, v + (max(times)*0.01), f'{v:.3f}s\n({times[0]/v:.1f}x)', ha='center', fontweight='bold')
    
    ax2.bar(scenarios, [tp / 1e6 for tp in throughputs], color=colors, edgecolor='black', alpha=0.8)
    ax2.set_ylabel('Throughput (Million pings/sec)', fontsize=12)
    ax2.set_title('Throughput Comparison (Higher is Better)', fontsize=13)
    for i, v in enumerate(throughputs):
        ax2.text(i, (v/1e6) + (max(throughputs)/1e6*0.01), f'{v/1e6:.2f} M/s', ha='center', fontweight='bold')
        
    plt.suptitle('PersistentJoiner Cache Reuse Performance Benefits', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

run_persistent_joiner_benchmark()

## Benchmark 4: Batch Size vs Throughput Analysis

In [ ]:
def run_batch_size_benchmark():
    print(f'\n=== Batch Size Optimization Analysis ===')
    n_pings_total = 50000000
    n_zones = 100_000
    res_target = 7
    base_index = 0x8928308280fffff

    # Data Generation
    print(f'Generating {n_pings_total:,} pings for batch testing...')
    # Use h3_turbo for fast data generation
    root_cell = np.array([base_index], dtype=np.uint64)
    pool_np = h3_turbo.grid_disk(root_cell, 200)
    pool_np = pool_np[pool_np != 0]
    zones = np.random.choice(pool_np, n_zones, replace=True)
    pings = pool_np[np.random.randint(0, len(pool_np), size=n_pings_total)]

    # Define batch sizes to test
    batch_sizes = [100_000, 500_000, 1_000_000, 5_000_000, 10_000_000, 25_000_000, 50_000_000]
    throughputs = []

    print(f"{'Batch Size':<15} | {'Throughput (M/s)':<20}")
    print('-' * 40)

    for batch_size in batch_sizes:
        # Force GC to clear previous run artifacts
        import gc; gc.collect()
        start_time = time.time()
        
        # Process in chunks
        for i in range(0, n_pings_total, batch_size):
            end = min(i + batch_size, n_pings_total)
            chunk = pings[i:end]
            # We use spatial_join as the workload
            _ = h3_turbo.spatial_join(chunk, zones, res_target, scramble_iterations=50)
            
        total_time = time.time() - start_time
        throughput = (n_pings_total / total_time) / 1_000_000
        throughputs.append(throughput)
        print(f"{batch_size:<15,} | {throughput:<20.2f}")

    # Save raw data to CSV
    import csv
    csv_fn = 'h3_turbo_benchmarks_batch_size.csv'
    with open(csv_fn, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Batch Size', 'Throughput (M/s)'])
        writer.writerows(zip(batch_sizes, throughputs))
    print(f'Raw data saved to {csv_fn}')

    # Plotting with Matplotlib
    plt.figure(figsize=(10, 6))
    plt.plot(batch_sizes, throughputs, marker='o', linestyle='-', color='#00aaff', linewidth=2)
    plt.xscale('log')
    plt.xlabel('Batch Size (Log Scale)', fontsize=12)
    plt.ylabel('Throughput (Million Ops/sec)', fontsize=12)
    plt.title('H3 Turbo: Batch Size vs Throughput', fontsize=14)
    plt.grid(True, which="both", ls="--", alpha=0.7)
    png_fn = 'h3_turbo_benchmarks_batch_size.png'
    plt.savefig(png_fn)
    print(f'Plot saved to {png_fn}')
    plt.show()

run_batch_size_benchmark()

## Benchmark 5: Accelerated Functions

In [ ]:
def run_accelerated_benchmarks(n=1000000):
    import platform
    import os
    try:
        num_workers = len(os.sched_getaffinity(0))
    except AttributeError:
        num_workers = os.cpu_count() or 1
    print(f'\n=== Accelerated Functions Benchmarks (N={n:,}) ===')
    print(f'CPU: {platform.processor()} with {num_workers} vCPUs')
    res = 9
    parent_res = 5
    k = 2
    
    # Generate Data
    lats = np.random.uniform(37.70, 37.81, n)
    lngs = np.random.uniform(-122.52, -122.35, n)
    
    # 1. latlng_to_cell
    print('\n--- latlng_to_cell ---')
    start = time.time()
    gpu_cells = h3_turbo.latlng_to_cell(lats, lngs, res)
    gpu_time = time.time() - start
    print(f'GPU Time: {gpu_time:.4f} s')
    
    start = time.time()
    # h3-py is scalar, use list comp
    cpu_cells = [h3.latlng_to_cell(lat, lng, res) for lat, lng in zip(lats, lngs)]
    cpu_time = time.time() - start
    print(f'CPU Time: {cpu_time:.4f} s')
    if gpu_time > 0: print(f'Speedup: {cpu_time/gpu_time:.2f}x')
    
    # 2. cell_to_parent
    print('\n--- cell_to_parent ---')
    start = time.time()
    gpu_parents = h3_turbo.cell_to_parent(gpu_cells, parent_res)
    gpu_time = time.time() - start
    print(f'GPU Time: {gpu_time:.4f} s')
    
    start = time.time()
    # Convert to string for h3-py
    cpu_cells_str = [h3.int_to_str(int(c)) for c in gpu_cells]
    cpu_parents = [h3.cell_to_parent(c, parent_res) for c in cpu_cells_str]
    cpu_time = time.time() - start
    print(f'CPU Time: {cpu_time:.4f} s')
    if gpu_time > 0: print(f'Speedup: {cpu_time/gpu_time:.2f}x')
    
    # 3. cell_to_boundary
    print('\n--- cell_to_boundary ---')
    # Use smaller N for boundary as it produces much more data
    n_bound = min(n, 100_000)
    gpu_cells_small = gpu_cells[:n_bound]
    cpu_cells_str_small = cpu_cells_str[:n_bound]
    
    start = time.time()
    gpu_bounds = h3_turbo.cell_to_boundary(gpu_cells_small)
    gpu_time = time.time() - start
    print(f'GPU Time ({n_bound:,} rows): {gpu_time:.4f} s')
    
    start = time.time()
    cpu_bounds = [h3.cell_to_boundary(c) for c in cpu_cells_str_small]
    cpu_time = time.time() - start
    print(f'CPU Time ({n_bound:,} rows): {cpu_time:.4f} s')
    if gpu_time > 0: print(f'Speedup: {cpu_time/gpu_time:.2f}x')
    
    # 4. grid_disk
    print('\n--- grid_disk (k=2) ---')
    start = time.time()
    gpu_disks = h3_turbo.grid_disk(gpu_cells_small, k)
    gpu_time = time.time() - start
    print(f'GPU Time ({n_bound:,} rows): {gpu_time:.4f} s')
    
    start = time.time()
    cpu_disks = [h3.grid_disk(c, k) for c in cpu_cells_str_small]
    cpu_time = time.time() - start
    print(f'CPU Time ({n_bound:,} rows): {cpu_time:.4f} s')
    if gpu_time > 0: print(f'Speedup: {cpu_time/gpu_time:.2f}x')
    
    # 5. grid_disk (scalar)
    print('\n--- grid_disk (k=2) (scalar) ---')
    n_kring = min(n_bound, 10_000)
    gpu_cells_kring = gpu_cells_small[:n_kring]
    cpu_cells_str_kring = cpu_cells_str_small[:n_kring]
    start = time.time()
    for c in gpu_cells_kring:
        _ = h3_turbo.grid_disk(c, k)
    gpu_time = time.time() - start
    print(f'GPU Time ({n_kring:,} ops): {gpu_time:.4f} s')
    
    start = time.time()
    for c in cpu_cells_str_kring:
        _ = h3.grid_disk(c, k) # h3-py equivalent
    cpu_time = time.time() - start
    print(f'CPU Time ({n_kring:,} ops): {cpu_time:.4f} s')
    if gpu_time > 0: print(f'Speedup: {cpu_time/gpu_time:.2f}x')

run_accelerated_benchmarks()

## Benchmark 6: 1 Billion Row Spatial Join (Q11)

In [ ]:
def run_1b_benchmark():
    n_pings = 1000000000
    n_zones = 10_000_000
    res_target = 7
    base_index = 0x8928308280fffff

    print(f'\n=== 1 Billion Row Spatial Join (Pings={n_pings:,}, Zones={n_zones:,}) ===')
    print('Generating Data...')
    # Use h3_turbo for fast data generation
    root_cell = np.array([base_index], dtype=np.uint64)
    pool_np = h3_turbo.grid_disk(root_cell, 200)
    pool_np = pool_np[pool_np != 0]
    zones = np.random.choice(pool_np, n_zones, replace=True)
    pings = pool_np[np.random.randint(0, len(pool_np), size=n_pings)]

    batch_size = 100_000_000

    # GPU Run
    print(f'Running GPU (Batch Size={batch_size:,})...')
    start_gpu = time.time()
    gpu_results = np.zeros(n_pings, dtype=np.uint8)
    for i in range(0, n_pings, batch_size):
        end = min(i + batch_size, n_pings)
        gpu_results[i:end] = h3_turbo.spatial_join(pings[i:end], zones, res_target, scramble_iterations=50)
    gpu_duration = time.time() - start_gpu
    print(f'GPU Time: {gpu_duration:.4f} s | Throughput: {n_pings/gpu_duration:,.0f} ops/s')

    # CPU Run (Multi-process)
    try:
        num_workers = len(os.sched_getaffinity(0))
    except AttributeError:
        num_workers = os.cpu_count() or 1
    import platform
    print(f'Running CPU (Multi-process on {num_workers} vCPUs, {platform.processor()}, Batch Size={batch_size:,})...')
    # Use h3_turbo for fast parent generation for CPU baseline setup
    zone_parents = numpy_apply_weight(h3_turbo.cell_to_parent(zones, res_target))
    zone_set = set(zone_parents)

    start_cpu = time.time()
    cpu_results = np.zeros(n_pings, dtype=np.uint8)
    chunk_size = 1_000_000
    with concurrent.futures.ProcessPoolExecutor(max_workers=num_workers, initializer=_init_worker, initargs=(zone_set,)) as executor:
        for i in range(0, n_pings, batch_size):
            end = min(i + batch_size, n_pings)
            batch_pings = pings[i:end]
            sub_chunks = (batch_pings[j:j + chunk_size] for j in range(0, len(batch_pings), chunk_size))
            results = list(executor.map(_process_chunk_cpu, sub_chunks, [res_target] * (len(batch_pings) // chunk_size)))
            cpu_results[i:end] = np.concatenate(results)
    cpu_duration = time.time() - start_cpu
    print(f'CPU Time: {cpu_duration:.4f} s')

    print(f'Speedup: {cpu_duration / gpu_duration:.2f}x')
    is_equal = np.array_equal(gpu_results, cpu_results)
    if not is_equal:
        print('ERROR: Results do not match!')
        print(f'GPU Sample: {gpu_results[:5]}')
        print(f'CPU Sample: {cpu_results[:5]}')
    assert is_equal, 'GPU and CPU results mismatch!'
    print('Verification Passed.')

run_1b_benchmark()